## PURPOSE
* Get the texts of attachments that is classified as ladung document

In [2]:
import os
import json
from ast import literal_eval
from datetime import datetime, timedelta

import pandas as pd
import numpy as np

from python_utilities.db_connection import DbConnection

In [7]:
attachment_ids = [
  "23448705574812-1",
  "23449089711516-1",
  "23449133228188-1",
  "23449198623004-1",
  "23449226050332-1",
  "23449240870812-1",
  "23449690808348-1",
  "23449977019036-1",
  "23451891332764-1",
  "23452682423452-1",
  "23452745861532-1",
  "23452946929180-1",
  "23453711916956-1",
  "23454531062428-1",
  "23454646129180-1"
]
len(attachment_ids)

15

In [4]:
analytics_db = DbConnection('ANALYTICS', 'PROD_RDS')

INFO [2025-11-10 13:02:09] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file


In [9]:
data = analytics_db.sql_to_df(f"""
    SELECT *
    FROM textract_jobs
    WHERE attachment_id IN {tuple(attachment_ids)}
    AND status = 'SUCCEEDED'
""")

# drop null values
data = data.dropna(subset=['s3_link'])
# drop dublicates
data = data.drop_duplicates(subset=['s3_link'])
s3_links = data['s3_link'].tolist()

In [10]:
data

,id,job_id,status,s3_link,created_at,updated_at,attachment_id
0,1138635,1261003399e820030e7f67509e369a89abc52fd3823e13...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 09:26:18,2025-11-07 09:52:39,23448705574812-1
1,1138727,ada380cf944310b2d1901a75bd5d7dd17c56dec3a8871b...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 10:14:52,2025-11-07 10:29:02,23449089711516-1
2,1138737,22679aea5ecd84678b7f8d6780fb8455c78070f87be8d0...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 10:14:58,2025-11-07 10:29:27,23449133228188-1
3,1138739,6c03efddc906587d2a909d61b6fcaf400f29c9963f8d24...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 10:14:59,2025-11-07 10:29:32,23449198623004-1
4,1138752,324275988bba61ee96a8b5061e8b081196a025ddebf413...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 10:15:06,2025-11-07 10:30:05,23449226050332-1
5,1138753,9e415e8edee139971005fb7f383ee940081ae7f7cb0063...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 10:15:06,2025-11-07 10:30:08,23449240870812-1
6,1138852,e2e0ca7759adf07195074916bd3f0ee7c14d70b5bbeba8...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 10:16:00,2025-11-07 10:34:22,23449690808348-1
7,1138877,1a30f040e1961d9e557b9c20898572ff7183f81f6c8d1b...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 10:16:14,2025-11-07 10:35:26,23449977019036-1
8,1139564,2d526f22c1c2355987ea6ed5d12d132248a2ff1e857747...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 12:14:10,2025-11-07 12:26:15,23451891332764-1
9,1139645,4c8b2d451cffdd6c2b11d564f338654e829b7277fcbb1a...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 12:14:59,2025-11-07 12:29:42,23452682423452-1


In [12]:
import boto3
def get_texts_from_textract_outputs(
        textract_jsons):
        return [
            "\n".join([d["Text"] for d in doc if d["BlockType"] == "LINE"])
            for doc in textract_jsons
        ]

def get_jsons_from_s3(s3_links):
    """
    Retrieves JSON objects from given S3 addresses.
    """
    # create S3 client with specific user profile
    session = boto3.Session(profile_name='739275445236_DataScienceUser')
    s3_client = session.client('s3')
    json_list = []

    for link in s3_links:
        json_object = [{"BlockType": "LINE", "Text": ""}]
        try:
            # Parse the S3 URI
            parts = link.replace("s3://", "").split("/")
            bucket_name = parts[0]
            key = "/".join(parts[1:])
            # Get the object from S3
            response = s3_client.get_object(Bucket=bucket_name, Key=key)

            # Read the content and parse JSON
            content = response['Body'].read().decode('utf-8')
            json_object = json.loads(content)
        except Exception as e:
            print(f"Error from {link} {e}")
            
        json_list.append(json_object)
    return json_list

returned_list=get_jsons_from_s3(s3_links)
texts = get_texts_from_textract_outputs(returned_list)

INFO [2025-11-10 13:05:34] - Found credentials in shared credentials file: ~/.aws/credentials


In [18]:
data['texts'] = texts

In [15]:
query = f"""
select *
from llm_attachments_predictions as lap1
where lap1.attachment_id in {tuple(attachment_ids)} and
subtype = 'aftercourt_parameters'
"""
attch_preds = analytics_db.sql_to_df(query)

In [16]:
attch_preds

,id,created_at,model_name,type,subtype,value,attachment_id
0,1608487,2025-11-07 11:18:35,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '184624280329', 'debtor_name': 'Erha...",23448705574812-1
1,1608733,2025-11-07 11:18:46,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '183615461781', 'debtor_name': 'Flor...",23449089711516-1
2,1608760,2025-11-07 11:18:47,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '179463008237', 'debtor_name': 'Tann...",23449133228188-1
3,1608770,2025-11-07 11:18:47,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': None, 'debtor_name': 'Alexander Hubr...",23449198623004-1
4,1608831,2025-11-07 11:18:50,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '178939667048', 'debtor_name': 'Aben...",23449226050332-1
5,1608837,2025-11-07 11:18:50,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '166008824087', 'debtor_name': 'Msri...",23449240870812-1
6,1609453,2025-11-07 11:19:18,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '180410205827', 'debtor_name': 'Alex...",23449690808348-1
7,1609638,2025-11-07 11:19:26,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '176302381031', 'debtor_name': 'Wikt...",23449977019036-1
8,1611894,2025-11-07 14:31:41,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '179380591612', 'debtor_name': 'Herr...",23451891332764-1
9,1612335,2025-11-07 14:32:01,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '178890222296', 'debtor_name': 'Marc...",23452682423452-1


In [19]:
attch_preds

,id,created_at,model_name,type,subtype,value,attachment_id
0,1608487,2025-11-07 11:18:35,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '184624280329', 'debtor_name': 'Erha...",23448705574812-1
1,1608733,2025-11-07 11:18:46,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '183615461781', 'debtor_name': 'Flor...",23449089711516-1
2,1608760,2025-11-07 11:18:47,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '179463008237', 'debtor_name': 'Tann...",23449133228188-1
3,1608770,2025-11-07 11:18:47,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': None, 'debtor_name': 'Alexander Hubr...",23449198623004-1
4,1608831,2025-11-07 11:18:50,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '178939667048', 'debtor_name': 'Aben...",23449226050332-1
5,1608837,2025-11-07 11:18:50,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '166008824087', 'debtor_name': 'Msri...",23449240870812-1
6,1609453,2025-11-07 11:19:18,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '180410205827', 'debtor_name': 'Alex...",23449690808348-1
7,1609638,2025-11-07 11:19:26,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '176302381031', 'debtor_name': 'Wikt...",23449977019036-1
8,1611894,2025-11-07 14:31:41,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '179380591612', 'debtor_name': 'Herr...",23451891332764-1
9,1612335,2025-11-07 14:32:01,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '178890222296', 'debtor_name': 'Marc...",23452682423452-1


In [21]:
data_text_and_preds = data.merge(attch_preds,
    on='attachment_id',
    how='left'
)

In [24]:
data_text_and_preds

,id_x,job_id,status,s3_link,created_at_x,updated_at,attachment_id,texts,id_y,created_at_y,model_name,type,subtype,value
0,1138635,1261003399e820030e7f67509e369a89abc52fd3823e13...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 09:26:18,2025-11-07 09:52:39,23448705574812-1,"Julia Reusch\nFrankfurter Straße 130, 2. Etage...",1608487,2025-11-07 11:18:35,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '184624280329', 'debtor_name': 'Erha..."
1,1138727,ada380cf944310b2d1901a75bd5d7dd17c56dec3a8871b...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 10:14:52,2025-11-07 10:29:02,23449089711516-1,Katja Fargel\nLindenweg 17\nObergerichtsvollzi...,1608733,2025-11-07 11:18:46,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '183615461781', 'debtor_name': 'Flor..."
2,1138737,22679aea5ecd84678b7f8d6780fb8455c78070f87be8d0...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 10:14:58,2025-11-07 10:29:27,23449133228188-1,Obergerichtsvollzieherin\nBüroanschrift (GV-Bü...,1608760,2025-11-07 11:18:47,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '179463008237', 'debtor_name': 'Tann..."
3,1138739,6c03efddc906587d2a909d61b6fcaf400f29c9963f8d24...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 10:14:59,2025-11-07 10:29:32,23449198623004-1,Obergerichtsvollzieherin\nBüroanschrift:\nSonj...,1608770,2025-11-07 11:18:47,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': None, 'debtor_name': 'Alexander Hubr..."
4,1138752,324275988bba61ee96a8b5061e8b081196a025ddebf413...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 10:15:06,2025-11-07 10:30:05,23449226050332-1,Obergerichtsvollzieherin\nHauptstraße 11\nDanj...,1608831,2025-11-07 11:18:50,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '178939667048', 'debtor_name': 'Aben..."
5,1138753,9e415e8edee139971005fb7f383ee940081ae7f7cb0063...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 10:15:06,2025-11-07 10:30:08,23449240870812-1,Obergerichtsvollzieherin\nHauptstraße 11\nDanj...,1608837,2025-11-07 11:18:50,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '166008824087', 'debtor_name': 'Msri..."
6,1138852,e2e0ca7759adf07195074916bd3f0ee7c14d70b5bbeba8...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 10:16:00,2025-11-07 10:34:22,23449690808348-1,Obergerichtsvollzieher\nBahnhofstraße 2-4\nFra...,1609453,2025-11-07 11:19:18,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '180410205827', 'debtor_name': 'Alex..."
7,1138877,1a30f040e1961d9e557b9c20898572ff7183f81f6c8d1b...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 10:16:14,2025-11-07 10:35:26,23449977019036-1,Obergerichtsvollzieherin\nBüroanschrift\nD. Sc...,1609638,2025-11-07 11:19:26,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '176302381031', 'debtor_name': 'Wikt..."
8,1139564,2d526f22c1c2355987ea6ed5d12d132248a2ff1e857747...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 12:14:10,2025-11-07 12:26:15,23451891332764-1,Obergerichtsvollzieher\nKaiserstraße 55\nMülle...,1611894,2025-11-07 14:31:41,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '179380591612', 'debtor_name': 'Herr..."
9,1139645,4c8b2d451cffdd6c2b11d564f338654e829b7277fcbb1a...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-11-07 12:14:59,2025-11-07 12:29:42,23452682423452-1,Obergerichtsvollzieherin\nBüroanschrift:\nSonj...,1612335,2025-11-07 14:32:01,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '178890222296', 'debtor_name': 'Marc..."


In [25]:
data_text_and_preds = data_text_and_preds[['attachment_id', 'texts', 'model_name','type','subtype','value']]

In [26]:
data_text_and_preds

,attachment_id,texts,model_name,type,subtype,value
0,23448705574812-1,"Julia Reusch\nFrankfurter Straße 130, 2. Etage...",aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '184624280329', 'debtor_name': 'Erha..."
1,23449089711516-1,Katja Fargel\nLindenweg 17\nObergerichtsvollzi...,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '183615461781', 'debtor_name': 'Flor..."
2,23449133228188-1,Obergerichtsvollzieherin\nBüroanschrift (GV-Bü...,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '179463008237', 'debtor_name': 'Tann..."
3,23449198623004-1,Obergerichtsvollzieherin\nBüroanschrift:\nSonj...,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': None, 'debtor_name': 'Alexander Hubr..."
4,23449226050332-1,Obergerichtsvollzieherin\nHauptstraße 11\nDanj...,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '178939667048', 'debtor_name': 'Aben..."
5,23449240870812-1,Obergerichtsvollzieherin\nHauptstraße 11\nDanj...,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '166008824087', 'debtor_name': 'Msri..."
6,23449690808348-1,Obergerichtsvollzieher\nBahnhofstraße 2-4\nFra...,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '180410205827', 'debtor_name': 'Alex..."
7,23449977019036-1,Obergerichtsvollzieherin\nBüroanschrift\nD. Sc...,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '176302381031', 'debtor_name': 'Wikt..."
8,23451891332764-1,Obergerichtsvollzieher\nKaiserstraße 55\nMülle...,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '179380591612', 'debtor_name': 'Herr..."
9,23452682423452-1,Obergerichtsvollzieherin\nBüroanschrift:\nSonj...,aftercourt_classification,aftercourt_classification,aftercourt_parameters,"'{'slug': '178890222296', 'debtor_name': 'Marc..."


In [27]:
# Extractions are not complete (due to bug). Do it again using current extraction
import os
print(os.getcwd())
os.chdir('../../')
print(os.getcwd())

from src.services.attachment_processing.aftercourt_extractors.ladung.strategy import LadungParameterExtractorStrategy
extractor = LadungParameterExtractorStrategy()

/Users/melih.gorgulu/Desktop/Projects/intent_recognition/notebooks/after-court
/Users/melih.gorgulu/Desktop/Projects/intent_recognition


In [28]:

from IPython.display import display, HTML

def show_colored_params(params, key_color="#ff6b6b", value_color="#4ecdc4"):
    if not params:
        display(
            HTML(
                "<div style='font-family:monospace;background:#1e1e1e;padding:12px;border-radius:8px;color:#ff6b6b;'>No parameters extracted.</div>"
            )
        )
        return

    content = "".join(
        f"<div><span style='color:{key_color};font-weight:600;'>{key}</span>: "
        f"<span style='color:{value_color};'>{value}</span></div>" for key, value in params.items()
    )
    display(
        HTML(
            "<div style='font-family:monospace;background:#1e1e1e;padding:12px;border-radius:8px;'>"
            + content
            + "</div>"
        )
    )
    
def print_colored_text(text: str, strings: list[str]) -> None:
    text = text.lower()
    colors = [
        "\033[91m",  # red
        "\033[92m",  # green
        "\033[94m",  # blue
        "\033[95m",  # magenta
        "\033[96m",  # cyan
        "\033[93m",  # yellow
    ]
    reset = "\033[0m"
    colored_text = text
    for idx, fragment in enumerate(strings):
        if not fragment:
            continue
        color = colors[idx % len(colors)]
        colored_text = colored_text.replace(fragment, f"{color}{fragment}{reset}")
    print(colored_text)

In [34]:
for index in range(len(data_text_and_preds)):
    print("INDEX = ", index)
    textract_text = data_text_and_preds.iloc[index]["texts"]
    #textract_text_translated = ladung_documents_all_data.iloc[index]["translated_text_TEXTRACT"]
    extracted_params = extractor.extract_aftercourt_parameters(textract_text)
    slug = extracted_params.get("slug", "")
    name, surname = extracted_params.get('debtor_name',' ').split(' ', 1) if 'debtor_name' in extracted_params else ('', '')
    name = name.lower()
    surname = surname.lower()
    date = extracted_params.get('judicial_summon_date', '')
    date = date.replace('-','.')
    show_colored_params(extracted_params)
    print("______________ATTACHMENT TEXT _____________")
    print_colored_text(textract_text, [slug, name, surname, date])
    print("_"*100)


INDEX =  0


______________ATTACHMENT TEXT _____________
julia reusch
frankfurter straße 130, 2. etage
gerichtsvollzieherin
51065 köln-buchheim
e-mail: julia.reusch@ag-koeln.nrw.de
sprechstunden:
dienstag 08:30-09:30 uhr
vom 24.11.-28.11., 02.12.-03.12. + 09.12.-
mittwoch 11-12 uhr
10.12. und 20.12. 02.01.2026 befinde ich
telefon 0176/22527158
mich im urlaub
egvp-nutzer-id für erv:
de.justiz.c8b31b62-d591-4eb7-93ce-
abs. gvin reusch, frankfurter straße 130, 2. etage, 51065 köln-
5f900f421a7f.bd85
buchheim
dienstkonto:
empfänger: anna julia reusch
firma
iban: de27 6609 0800 0008 3255 53
pair finance gmbh
bic: genode61bbb
knesebeckstraße 62-63
bank: bbbank karlsruhe
10719 berlin
dr ii 1672/25
bitte bei schreiben und zahlungen angeben!
184624280329
köln-buchheim, 07.11.2025
zwangsvollstreckungssache
firma liquandum capital ii gmbh, knesebeckstraße 62-63, 10719 berlin, az.184624280329
vertreten durch: firma pair finance gmbh, knesebeckstraße 62-63, 10719 berlin, az.184624280329
gegen
herrn erhan unver,

______________ATTACHMENT TEXT _____________
katja fargel
lindenweg 17
obergerichtsvollzieherin
27299 langwedel
e-mail: gvin.katja.fargel@gerichtsvollzieher.de
sprechstunden:
dienstag 12:00-14:00 uhr
mein büro ist vom 03.11.2025-21.11.2025
mittwoch
09:00 -10:00 uhr
geschlossen!
telefon 04232-934646
egvp-nutzer-id für erv:
abs. ogv'in fargel, lindenweg 17, 27299 langwedel
safe-sp1-1357997481418-012162952
dienstkonto:
pair finance gmbh, vertr.d.d. gf
empfänger: katja fargel
knesebeckstraße 62-63
iban: de31 2905 0101 0001 0500 87
10719 berlin
bic: sbrede22xxx
bank: spk bremen
dr ii 1271/25
bitte bei schreiben und zahlungen angeben!
183615461781
bremen, 07.11.2025
zwangsvollstreckungssache
liquandum capital ii gmbh, vertr.d.d. gf, knesebeckstraße 62-63, 10719 berlin ot wilmersdorf
vertreten durch: pair finance gmbh, vertr.d.d. gf, knesebeckstraße 62-63, 10719 berlin, az.183615461781
gegen
herrn florian fleddermann, schwarzer weg 15, 28239 bremen ot ohlenhof
sehr geehrte damen und herren,
ge

______________ATTACHMENT TEXT _____________
obergerichtsvollzieherin
büroanschrift (gv-büro 1) eg
christine hoff
höhenstr. 74, 40227 düsseldorf (linie 706,704)
bürozeiten
dienstag: 12.00 13.00 uhr
donnerstag: 08.00 09.00 uhr
telefon: 0211/ 73774429 (nur i.d. bürozeiten)
handy:0178/5315150 (unregelmässig)
e-mail
christine.hoff@ag-duesseldorf.nrw.de
abs.: ogvin hoff, höhenstr. 74, 40227 düsseldorf
dienstkonto:
iban de66330400010292825700
pair finance gmbh vertr. d. d. gf
bic cobadeff330
knesebeckstraße 62-63
empfänger: christine hoff
10719 berlin
mein zeichen
ihr zeichen
dr ii 1626/25
179463008237
düsseldorf, den 07.11.2025
bitte immer angeben!
zwangsvollstreckungssache
e.on energie deutschland gmbh, arnulfstraße 203, 80634 münchen
vertr. d.
pair finance gmbh vertr. d. d. gf, knesebeckstraße 62-63, 10719 berlin, tel. 030-340602950, e-mail
aftercourt@pairfinance.de
gegen
tannaz alizadeh milajerdi, poßbergweg 69, 40629 düsseldorf
sehr geehrte damen und herren,
in o.g. sache habe ich gemäß 

______________ATTACHMENT TEXT _____________
obergerichtsvollzieherin
büroanschrift:
sonja weber
roisdorfer str. 14/ecke bornheimer straße
50969 köln
amtsgericht
köln
bürozeiten/sprechzeiten:
mo.+ mi. 14.00-15.00 uhr
0221/9362507 (nur zur sprechzeit!)
telefon
02645/972314 (ausserhalb sprechzeit)
e-mail
abs.: ogvin sonja weber, roisdorfer str. 14, 50969 köln
sonja.weber@ag-koeln.nrw.de
falls empfänger verzogen, bitte an absender zurück!
safe-id:
pair finance gmbh
safe-sp1-1352189778363-011604190
knesebeckstraße 62-63
dienstkonto
10719 berlin
volksbank köln bonn eg
iban de62380601868715940010
bic genoded1brs
mein zeichen
ihr zeichen
dr ii 1627/25
köln, 07.11.2025
bitte immer angeben!
das büro ist vom 11.11.25 bis vorraussichtlich 29.11.25 geschlossen!
zwangsvollstreckungssache
pair finance gmbh, knesebeckstraße 62-63, 10719 berlin, e-mail aftercourt@pairfinance.de
vertr. d.
firma liquandum capital il gmbh, knesebeckstraße 62-63, 10719 berlin
gegen
herrn alexander hubrich, siebengebirgsall

...

______________ATTACHMENT TEXT _____________
obergerichtsvollzieherin andrea steiger
amtsgericht sonthofen
personenbezogene daten werden verarbeitet. informationen erhalten sie auf der internetseite des amtsgerichts sonthofen: https://www.justiz.bayern.de/gerichte-und-
behoerden/amtsgerichte/sonthofen//info_service_1.php unter "datenschutz"
dienstkonto:
hirschstraße 10 (1. stock)
tel. 08321 / 6741880
bbbank karlsruhe eg
87527 sonthofen
fax 08321 / 6740742
iban: per lastschrift!
mobil 0170 / 9248836
bic: genode61bbb
sprechzeiten im büro:
dienstag 14 15 uhr
abs.: ogvin a. steiger, hirschstraße 10, 87527 sonthofen
donnerstag 10 11 uhr
firma
e-mail:
pair finance gmbh
ogvin-steiger@gvpost.de
vertr. d.d. gf
egvp-postfach amtsgericht sonthofen:
knesebeckstraße 62-63
de.justiz.e339d7db-ad37-4c1a-b3b1-c70e60471e0b.8bf2
10719 berlin
mein zeichen
ihr zeichen
3 dr 274/25
174128539536
sonthofen, 07.11.2025
bitte immer angeben!
zwangsvollstreckungssache
sehr geehrte damen und herren,
in o.g. sache ha

______________ATTACHMENT TEXT _____________
obergerichtsvollzieherin michaela posch
telefon:
sprechstunden/büro:
dienstkonto
0208/69819680
di und do
iban de93360501050003054301
9.00 uhr bis 10 uhr
bic spesde3exxx
clausewitzstr. 22
sparkasse essen
email:
45472 mülheim/ruhr
gv.posch@arcor.de
nur postanschrift: clausewitzstraße 22/24, 45472 mülheim/ruhr
abs.ogvin michaela posch,clausewitzstr.22/24 45472 mülheim
pair finance gmbh
knesebeckstraße 62-63
10719 berlin
mein zeichen
ihr zeichen
dr ii 1879/25
173986555519
mülheim a. d. ruhr, 07.11.2025
bitte immer angeben!
zwangsvollstreckungssache
e.on energie deutschland gmbh, arnulfstraße 203, 80634 münchen
vertr. d.
pair finance gmbh, knesebeckstraße 62-63, 10719 berlin, e-mail aftercourt@pairfinance.de
gegen
frau ghazale al-malla, aktienstraße 279, 45473 mülheim an der ruhr
sehr geehrte damen und herren,
in o. g. sache habe ich gemäß ihrem auftrag termin zur abgabe der vermögensauskunft auf montag, 01.12.25,
08:30 uhr, büro 45472 mülheim, cl

______________ATTACHMENT TEXT _____________
n. becker
bunzlauer str. 19/rgb.
gerichtsvollzieherin
80992 münchen
e-mail: gvin.becker@gvzentrale.de
sprechstunden:
mo.: 11-13 uhr, do.: 10-12 uhr
telefon 089/215390996
telefax 089/215390997
abs. gvin becker, bunzlauer str. 19/rgb., 80992 münchen
egvp-nutzer-id für erv:
firma
dienstkonto:
pair finance gmbh
empfänger: gvin becker
knesebeckstraße 62-63
iban: de13660908000005914191
10719 berlin
bic: genode61bbb
bank: bbbank eg karlsruhe
31 dr ii 1767/25
bitte bei schreiben und zahlungen angeben!
179844448524
münchen, 07.11.2025
zwangsvollstreckungssache
firma liquandum capital ii gmbh, vertr. d. d. gf, knesebeckstraße 62-63, 10719 berlin
vertreten durch: firma pair finance gmbh, knesebeckstraße 62-63, 10719 berlin, az.179844448524
gegen
herrn oliver christian edgar bieber, agnes-bernauer-straße 18, 80687 münchen
sehr geehrte damen und herren,
gem. ihrem antrag habe ich einen termin zur abgabe der vermögensauskunft auf
montag, 1. dezember 2025, 

______________ATTACHMENT TEXT _____________
obergerichtsvollzieherin am amtsgericht hannover
büroanschrift:
eva döbel
hainhölzer straße 5 (grüne tür)
30159 hannover
bürozeiten:
dienstags 09:30- 10:30 uhr
donnerstags 11:30- 12:30 uhr
telefon: 0511-45081170 z.d. bürozeiten
05128-221465 homeoffice
e-mail: ogv.e.doebel@gvpost.de
dienstkonto: commerzbank hannover
kontobezeichnung: eva maria lydia döbel
ogv döbel, hainhölzer straße 5, 30159 hannover
de 51 2504 0066 0161 9535 00
cobadeffxxx
firma
pair finance gmbh
vertr.d.d. gf
knesebeckstraße 62-63
10719 berlin
mein zeichen
ihr zeichen
dr ii 1047/25
179643299912
hannover, 07.11.2025
bitte immer angeben!
zwangsvollstreckungssache
firma liquandum capital ii gmbh, knesebeckstr. 62-63, 10719 berlin
vertr. d.
firma pair finance gmbh, knesebeckstraße 62-63, 10719 berlin
gegen
herrn alireza hajizadeh, thomastraße 3, 30177 hannover
sehr geehrte damen und herren,
in o.g. sache habe ich gemäß ihrem auftrag termin zur abgabe der vermögensauskunft auf d

______________ATTACHMENT TEXT _____________
sascha hampel
wilhelmstraße 6
obergerichtsvollzieher
52428 jülich
e-mail: gvhampel@web.de
sprechstunden:
montag 13:30 uhr 14:45 uhr
mittwoch 09:00 uhr 10:00 uhr
telefon 022525489857
abs. ogv hampel, wilhelmstraße 6, 52428 jülich
egvp-nutzer-id für erv:
safe-sp1-1487080547485-016416258
firma
dienstkonto:
pair finance gmbh
empfänger: sascha hampel
knesebeckstraße 62-63
iban: de09395501101201052493
10719 berlin
bic: sduede33xxx
bank: sparkasse düren
dr ii 2069/25
bitte bei schreiben und zahlungen angeben!
180635948342
jülich, 04.11.2025
zwangsvollstreckungssache
firma liquandum capital gmbh, knesebeckstraße 62-63, 10719 berlin
vertreten durch: firma pair finance gmbh, knesebeckstraße 62-63, 10719 berlin, az.180635948342
gegen
herrn arjan sejdiu, mühlenhecke 4, 52382 niederzier-huchem-stammeln
sehr geehrte damen und herren,
gem. ihrem antrag habe ich einen termin zur abgabe der vermögensauskunft auf
montag, 8. dezember 2025, um 14:45 uhr,
büro, w